# Week 12A — Stereo Vision and Point Clouds

**MCTA 4364 Machine Vision** · Session 1 of Week 12 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W12_3d_vision/W12A_stereo_point_clouds.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W12_3d_vision/W12A_stereo_point_clouds.ipynb)

A single camera loses depth: everything along a ray projects to the same pixel (Week 2). Robots that pick parts from a bin, inspection of
heights, volumes and flatness, and navigation all need **3D**. This session recovers depth the way our two eyes do: **two calibrated cameras**,
**epipolar geometry**, **stereo matching** and **triangulation**, and turns depth maps into **point clouds** that can be measured. It ends with a
height-inspection station on the conveyor.

### Learning outcomes
By the end of this session you will be able to:
1. Compare 3D sensing technologies (passive/active stereo, structured light, time-of-flight, laser triangulation).
2. Explain **epipolar geometry**, the fundamental matrix, **stereo calibration** and **rectification**.
3. Compute disparity with block matching and **SGBM**, evaluate it against ground truth, and explain its failure modes.
4. Convert disparity to depth ($Z = fB/d$), predict the **depth resolution**, and back-project to a coloured **point cloud**.
5. Fit a plane with **RANSAC** and measure object heights; explain why **active** stereo projects a pattern.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1 | 3D sensors |
| 8–25 | 2 | Epipolar geometry; stereo calibration and rectification |
| 25–45 | 3–4 | Disparity: matching costs, SGBM (interactive), accuracy; depth resolution |
| 45–60 | 5 | Point clouds (interactive 3D) |
| 60–72 | 6 | Height inspection on the conveyor: plane fitting, active stereo |
| 72–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

**Hardware:** laptop CPU (about 1 minute). Data: OpenCV sample stereo images (a few MB).

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

import partsim
from cvhelpers import SMOKE, check, interact, load_sample, pipeline_diagram, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
aloeL, aloeR = load_sample("aloeL.jpg"), load_sample("aloeR.jpg")
aloeGT = load_sample("aloeGT.png", gray=True).astype(np.float32)          # ground-truth disparity (px, full resolution; 0 = unknown)
S = 0.5                                                                   # work at half resolution for speed
L = cv2.resize(aloeL, None, fx=S, fy=S, interpolation=cv2.INTER_AREA)
R = cv2.resize(aloeR, None, fx=S, fy=S, interpolation=cv2.INTER_AREA)
GT = cv2.resize(aloeGT, None, fx=S, fy=S, interpolation=cv2.INTER_NEAREST) * S
show(L, R, GT, titles=["aloe left (Middlebury 2006)", "aloe right", "ground-truth disparity (black = unknown)"], cmap="magma", figsize=(20, 5))

## 1. How machines sense depth

| technology | principle | strengths | weaknesses | examples |
|---|---|---|---|---|
| **passive stereo** | two cameras, triangulate matched pixels | cheap, outdoor, any range | needs texture; heavy computation | ZED, OAK-D |
| **active stereo** | stereo + a projected random **speckle** pattern | works on textureless surfaces | pattern washed out by sunlight | Intel RealSense D4xx |
| **structured light** | project known coded patterns, one camera | very accurate at short range | slow (several patterns) or sensitive to motion | Zivid, Photoneo, Kinect v1 |
| **time-of-flight (ToF)** | measure the light's round-trip time/phase per pixel | dense, fast, no texture needed | multipath, low resolution | Kinect Azure, Basler blaze |
| **laser triangulation** | a laser line + camera, the object moves under it | micrometre precision, inline | one profile at a time | Keyence LJ, SICK Ranger |
| **LiDAR** | scanning laser time-of-flight | long range, outdoor | sparse, expensive | vehicles, mobile robots |

All triangulation-based methods share the geometry of this session.

## 2. Epipolar geometry, calibration and rectification

A point $\mathbf{x}$ in the left image can only match a point on a **line** in the right image — its **epipolar line** $\mathbf{l}' = F\mathbf{x}$ —
because the 3D point must lie on the ray through $\mathbf{x}$. The **fundamental matrix** $F$ (3×3, rank 2) encodes this:
$\mathbf{x}'^\top F \mathbf{x} = 0$. It can be estimated from 8+ matches with RANSAC (Week 5B).

**Stereo calibration** estimates both cameras' intrinsics (Week 2B) and the rotation $R$ and translation $\mathbf{t}$ between them.
**Rectification** then warps both images so that epipolar lines become **horizontal and aligned rows**: matching becomes a 1D search along a row.
For rectified cameras with focal length $f$ (px) and baseline $B$:

$$Z = \frac{f\,B}{d}, \qquad d = x_\text{left} - x_\text{right} \;\text{(disparity)}.$$

We calibrate OpenCV's stereo chessboard pairs (`left01…14`, `right01…14`).

In [ ]:
PATTERN = (9, 6)
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)            # units = one chessboard square
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
obj_pts, left_pts, right_pts, pairs = [], [], [], []
for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14][: 5 if SMOKE else 13]:
    gl, gr = load_sample(f"left{i:02d}.jpg", gray=True), load_sample(f"right{i:02d}.jpg", gray=True)
    ok1, c1 = cv2.findChessboardCorners(gl, PATTERN)
    ok2, c2 = cv2.findChessboardCorners(gr, PATTERN)
    if ok1 and ok2:
        c1 = cv2.cornerSubPix(gl, c1.reshape(-1, 1, 2).astype(np.float32), (11, 11), (-1, -1), crit)   # OpenCV 5 returns (N, 2)
        c2 = cv2.cornerSubPix(gr, c2.reshape(-1, 1, 2).astype(np.float32), (11, 11), (-1, -1), crit)
        obj_pts.append(objp); left_pts.append(c1); right_pts.append(c2); pairs.append((gl, gr))
size = pairs[0][0].shape[::-1]
_, K1, D1, _, _ = cv2.calibrateCamera(obj_pts, left_pts, size, None, None)
_, K2, D2, _, _ = cv2.calibrateCamera(obj_pts, right_pts, size, None, None)
rms, K1, D1, K2, D2, Rst, Tst, E, Fm = cv2.stereoCalibrate(obj_pts, left_pts, right_pts, K1, D1, K2, D2, size, flags=cv2.CALIB_FIX_INTRINSIC)
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(K1, D1, K2, D2, size, Rst, Tst, alpha=0)
print(f"{len(pairs)} stereo pairs | stereo reprojection RMS {rms:.3f} px | baseline |t| = {np.linalg.norm(Tst):.2f} squares | "
      f"rectified focal length {P1[0, 0]:.1f} px")

In [ ]:
gl, gr = pairs[0]
# epipolar lines of a few left-image corners, drawn in the RIGHT image, before rectification
pts_l = left_pts[0][::11].reshape(-1, 2)
lines = cv2.computeCorrespondEpilines(pts_l.reshape(-1, 1, 2), 1, Fm).reshape(-1, 3)
vis_r = cv2.cvtColor(gr, cv2.COLOR_GRAY2BGR); vis_l = cv2.cvtColor(gl, cv2.COLOR_GRAY2BGR)
cols = [tuple(int(c) for c in np.array(plt.cm.tab10(k)[:3]) * 255) for k in range(10)]
for k, ((a, b, c), p) in enumerate(zip(lines, pts_l)):
    x0, x1 = 0, vis_r.shape[1]
    cv2.line(vis_r, (x0, int(-c / b)), (x1, int(-(c + a * x1) / b)), cols[k % 10], 2)
    cv2.circle(vis_l, tuple(p.astype(int)), 6, cols[k % 10], -1)
m1 = cv2.initUndistortRectifyMap(K1, D1, R1, P1, size, cv2.CV_32FC1)
m2 = cv2.initUndistortRectifyMap(K2, D2, R2, P2, size, cv2.CV_32FC1)
rl, rr = cv2.remap(gl, *m1, cv2.INTER_LINEAR), cv2.remap(gr, *m2, cv2.INTER_LINEAR)
rect = cv2.cvtColor(np.hstack([rl, rr]), cv2.COLOR_GRAY2BGR)
for y in range(20, rect.shape[0], 40):
    cv2.line(rect, (0, y), (rect.shape[1], y), (0, 200, 0), 1)
fig, axes = plt.subplots(1, 3, figsize=(24, 5), gridspec_kw=dict(width_ratios=[1, 1, 2]))
axes[0].imshow(vis_l[..., ::-1]); axes[0].set_title("left: selected corners")
axes[1].imshow(vis_r[..., ::-1]); axes[1].set_title("right: their epipolar lines (slanted)")
axes[2].imshow(rect[..., ::-1]); axes[2].set_title("after rectification: corresponding points share the same row (green lines)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 2.1 - Epipolar lines before and after stereo rectification", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Stereo matching

On a rectified pair, the match of left pixel $(x, y)$ is at $(x - d, y)$ for some disparity $d \in [d_\min, d_\max]$. **Block matching** compares a
window around each pixel with windows along the row using a cost such as SAD, SSD or census, and picks the minimum. It fails on **textureless**
regions (many equal costs), **repetitive** texture (several minima), **occlusions** (the point is hidden in one view) and depth discontinuities.

**Semi-Global Matching** (SGM, Hirschmüller, 2008; `cv2.StereoSGBM`) adds a smoothness term along 4–8 image directions: penalty $P_1$ for disparity
changes of 1 px and $P_2 > P_1$ for larger jumps. It is the workhorse of industrial and automotive stereo. Disparities are sub-pixel (1/16 px).

In [ ]:
gL, gR = cv2.cvtColor(L, cv2.COLOR_BGR2GRAY).astype(np.float32), cv2.cvtColor(R, cv2.COLOR_BGR2GRAY).astype(np.float32)
fig, axes = plt.subplots(1, 3, figsize=(22, 4.5), gridspec_kw=dict(width_ratios=[1, 1, 1.4]))
local_std = np.sqrt(np.maximum(cv2.blur(gL ** 2, (11, 11)) - cv2.blur(gL, (11, 11)) ** 2, 0))
cand = GT > 0; cand[:20] = cand[-20:] = False; cand[:, :130] = cand[:, -20:] = False
yu, xu = np.unravel_index(np.argmin(np.where(cand, local_std, 1e9)), local_std.shape)      # the least textured valid pixel
examples = [(300, 250, "textured leaf"), (int(xu), int(yu), "least textured spot")]
for k, (x, y, name) in enumerate(examples):
    win = 5
    ref = gL[y - win:y + win + 1, x - win:x + win + 1]
    ds = np.arange(0, 110)
    cost = [np.abs(ref - gR[y - win:y + win + 1, x - d - win:x - d + win + 1]).sum() if x - d - win >= 0 else np.nan for d in ds]
    axes[2].plot(ds, cost, lw=2, label=f"{name} (true d = {GT[y, x]:.1f})")
    axes[k].imshow(L[..., ::-1]); axes[k].add_patch(plt.Rectangle((x - win, y - win), 2 * win + 1, 2 * win + 1, fill=False, ec="#fbbc05", lw=2))
    axes[k].set_title(f"{name} at ({x}, {y})"); axes[k].axis("off")
axes[2].set_xlabel("disparity candidate d (px)"); axes[2].set_ylabel("SAD cost of an 11 x 11 window"); axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].set_title("texture gives a clear minimum; without texture the minimum can be at the wrong disparity")
plt.suptitle("Figure 3.1 - The matching cost along the epipolar line", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def disparity(method="SGBM", num_disp=128, block=5, uniqueness=10):
    if method == "BM":
        st = cv2.StereoBM_create(numDisparities=num_disp, blockSize=max(5, block | 1))
        st.setUniquenessRatio(uniqueness)
        return st.compute(gL.astype(np.uint8), gR.astype(np.uint8)).astype(np.float32) / 16
    st = cv2.StereoSGBM_create(0, num_disp, block, P1=8 * 3 * block ** 2, P2=32 * 3 * block ** 2, uniquenessRatio=uniqueness,
                               speckleWindowSize=100, speckleRange=2, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
    return st.compute(L, R).astype(np.float32) / 16


def evaluate(d):
    valid = (GT > 0) & (d > 0)
    err = np.abs(d - GT)[valid]
    return dict(density=valid.sum() / (GT > 0).sum(), bad2=(err > 2).mean(), epe=err.mean())


results = {}
for name in ["BM", "SGBM"]:
    t0 = time.perf_counter(); d = disparity(name); dt = time.perf_counter() - t0
    results[name] = (d, evaluate(d), dt)
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
axes[0].imshow(GT, cmap="magma", vmin=0, vmax=GT.max()); axes[0].set_title("ground truth")
for ax, (name, (d, ev, dt)) in zip(axes[1:], results.items()):
    ax.imshow(np.where(d > 0, d, np.nan), cmap="magma", vmin=0, vmax=GT.max())
    ax.set_title(f"{name}: density {ev['density']:.0%}, bad pixels (> 2 px) {ev['bad2']:.1%}, mean error {ev['epe']:.2f} px ({dt * 1000:.0f} ms)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 3.2 - Block matching vs semi-global matching (white/transparent = no valid match)", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: SGBM parameters

In [ ]:
def sgbm_demo(block=5, num_disp=128, uniqueness=10):
    d = disparity("SGBM", num_disp, block, uniqueness)
    ev = evaluate(d)
    show(np.where(d > 0, d, 0), titles=[f"block {block}, numDisparities {num_disp}, uniqueness {uniqueness}: density {ev['density']:.0%}, "
                                        f"bad {ev['bad2']:.1%}, error {ev['epe']:.2f} px"], cmap="magma", figsize=(9, 7))


interact(sgbm_demo, block=widgets.IntSlider(value=5, min=1, max=15, step=2), num_disp=widgets.IntSlider(value=128, min=16, max=192, step=16),
         uniqueness=widgets.IntSlider(value=10, min=0, max=30))

## 4. From disparity to depth, and depth resolution

Differentiating $Z = fB/d$: a disparity error $\Delta d$ gives a depth error

$$\Delta Z \approx \frac{Z^2}{f\,B}\,\Delta d.$$

Depth error grows with the **square** of the distance and shrinks with a longer baseline or focal length. A longer baseline, however, increases
occlusions and the disparity search range. This is the first calculation when specifying a 3D camera.

In [ ]:
Z = np.linspace(200, 3000, 200)                          # mm
fig, ax = plt.subplots(figsize=(11, 4.2))
for f_px, B_mm, col in [(700, 50, "#d93025"), (700, 120, "#f29900"), (1400, 120, "#188038")]:
    ax.plot(Z, Z ** 2 / (f_px * B_mm) * 0.25, lw=2.5, color=col, label=f"f = {f_px} px, B = {B_mm} mm")
ax.set_xlabel("distance Z (mm)"); ax.set_ylabel("depth error for a 0.25 px disparity error (mm)"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Figure 4.1 - Depth resolution: quadratic in distance, inverse in baseline and focal length", fontweight="bold")
plt.show()

## 5. Point clouds

Each valid pixel $(u, v)$ with depth $Z$ back-projects through the pinhole model (Week 2) to a 3D point:

$$X = \frac{(u - c_x)\,Z}{f}, \qquad Y = \frac{(v - c_y)\,Z}{f}, \qquad Z = \frac{fB}{d}.$$

The Middlebury aloe pair was taken with $f \approx 3740$ px and $B = 160$ mm at full resolution (so $f \approx 1870$ px at our half resolution).
The images were **cropped** after rectification, so a constant offset $d_\min$ (270 px at full resolution for aloe) must be added:
$Z = fB / (d + d_\min)$ — a detail that is easy to miss when using public datasets.
The result, with colours, is a **point cloud**; libraries such as **Open3D** and **PCL** then filter, register, segment and mesh it.

In [ ]:
f_half, B_mm, DMIN = 3740.0 * S, 160.0, 270.0 * S
d_sgbm = results["SGBM"][0]
h_, w_ = d_sgbm.shape
v_, u_ = np.mgrid[0:h_, 0:w_]
ok = d_sgbm > 5
Zc = f_half * B_mm / np.where(ok, d_sgbm + DMIN, 1)
pts = np.stack([(u_ - w_ / 2) * Zc / f_half, (v_ - h_ / 2) * Zc / f_half, Zc], -1)[ok]
cols = L[..., ::-1][ok]
step = 40 if SMOKE else 8
sel = np.arange(len(pts))[::step]
fig = go.Figure(go.Scatter3d(x=pts[sel, 0], y=pts[sel, 2], z=-pts[sel, 1], mode="markers",
                             marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in cols[sel]])))
fig.update_layout(height=520, margin=dict(l=0, r=0, t=40, b=0), title="Figure 5.1 - Interactive coloured point cloud (drag to rotate)",
                  scene=dict(xaxis_title="X (mm)", yaxis_title="Z depth (mm)", zaxis_title="-Y (mm)", aspectmode="data"))
fig.show()
print(f"{ok.sum():,} points, depth range {Zc[ok].min():.0f}-{np.percentile(Zc[ok], 99):.0f} mm (plotted every {step}th point)")

## 6. Application: height inspection on the conveyor

A stereo camera looks straight down at the belt from $Z_0 = 500$ mm ($f = 1000$ px, $B = 60$ mm, so the belt has $d = 120$ px). Each part has a
nominal height (nut 8 mm, washer 2 mm, bolt 10 mm, flange 5 mm). We **simulate** the rectified stereo pair from the part heights, match it with SGBM,
fit the **belt plane with RANSAC** (Week 5B) and measure every part's height above it.

Two lessons: (1) on a **smooth** belt passive stereo has nothing to match, so we project a random **speckle** pattern — **active stereo**
(the two cameras get independent sensor noise, as real cameras do); (2) a 2 mm washer changes the disparity by only ~0.5 px: close to the
resolution limit of Section 4.

In [ ]:
F_PX, B_BELT, Z0 = 1000.0, 60.0, 500.0
HEIGHT = {"hex_nut": 8.0, "washer": 2.0, "bolt": 10.0, "flange": 5.0}
scene, sb, sl, smasks = partsim.make_scene(7, return_masks=True, noise=0)      # noise is added per camera below
height = np.zeros(scene.shape[:2], np.float32)
for m, c in zip(smasks, sl):
    height[m] = HEIGHT[partsim.SCENE_CLASSES[c]]
on_part = height > 0
smooth_scene = scene.copy()
smooth_scene[~on_part] = cv2.GaussianBlur(scene, (0, 0), 8)[~on_part]           # a smooth, polished belt


def render_stereo(img, height, speckle=True, seed=0):
    """Rectified stereo pair of a scene whose surface is `height` mm above the belt (camera looking straight down)."""
    left = img.astype(np.float32)
    if speckle:                                             # active stereo: a projected random dot pattern
        dots = (np.random.default_rng(seed).random(img.shape[:2]) < 0.08).astype(np.float32)
        left = left + cv2.GaussianBlur(dots, (0, 0), 0.8)[..., None] * 120
    d_left = F_PX * B_BELT / (Z0 - height)
    hh, ww = height.shape
    xx, yy = np.meshgrid(np.arange(ww, dtype=np.float32), np.arange(hh, dtype=np.float32))
    d_right = np.full_like(d_left, F_PX * B_BELT / Z0)
    for _ in range(3):                                      # disparity seen from the right camera (fixed-point iteration)
        d_right = cv2.remap(d_left, xx + d_right, yy, cv2.INTER_NEAREST, borderMode=cv2.BORDER_REPLICATE)
    right = cv2.remap(left, xx + d_right, yy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    rng = np.random.default_rng(seed + 1)
    left, right = left + rng.normal(0, 3, left.shape), right + rng.normal(0, 3, right.shape)     # independent sensor noise
    return np.clip(left, 0, 255).astype(np.uint8), np.clip(right, 0, 255).astype(np.uint8)


def measure_heights(left, right):
    st = cv2.StereoSGBM_create(96, 64, 3, P1=2 * 3 * 9, P2=8 * 3 * 9, uniquenessRatio=5, mode=cv2.STEREO_SGBM_MODE_HH)
    d = st.compute(left, right).astype(np.float32) / 16
    valid = d > 96
    return np.where(valid, Z0 - F_PX * B_BELT / np.where(valid, d, 1), np.nan)


conditions = [("textured belt, passive", scene, False), ("smooth belt, passive", smooth_scene, False), ("smooth belt, active (speckle)", smooth_scene, True)]
h_est = {name: measure_heights(*render_stereo(img_, height, speckle=sp)) for name, img_, sp in conditions}
belt_px = ~on_part; belt_px[:, :170] = False                                   # belt pixels inside the search range
left_a, right_a = render_stereo(smooth_scene, height, True)
fig, axes = plt.subplots(1, 5, figsize=(26, 4.4))
axes[0].imshow(left_a[..., ::-1]); axes[0].set_title("smooth belt + projected speckle (left)")
axes[1].imshow(height, cmap="viridis", vmin=0, vmax=11); axes[1].set_title("true height (mm)")
for ax, (name, he) in zip(axes[2:], h_est.items()):
    ok_b = np.isfinite(he[belt_px])
    wrong = (np.abs(he[belt_px][ok_b]) > 1).mean() if ok_b.any() else 1.0
    im_ = ax.imshow(he, cmap="viridis", vmin=0, vmax=11); ax.set_title(f"{name}\nbelt: {ok_b.mean():.0%} valid, {wrong:.0%} of those off by > 1 mm", fontsize=9)
plt.colorbar(im_, ax=axes[4], fraction=0.035, label="mm")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.suptitle("Figure 6.1 - Stereo height maps of the conveyor (left band: outside the disparity search range)", fontweight="bold", y=1.06)
plt.show()

In [ ]:
def fit_plane_ransac_ref(points, thr=0.5, iters=200, seed=0):
    rng = np.random.default_rng(seed)
    best_n, best_c, best_in = None, None, np.zeros(len(points), bool)
    for _ in range(iters):
        p = points[rng.choice(len(points), 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0])
        if np.linalg.norm(n) < 1e-9:
            continue
        n = n / np.linalg.norm(n); c = -n @ p[0]
        inl = np.abs(points @ n + c) < thr
        if inl.sum() > best_in.sum():
            best_n, best_c, best_in = n, c, inl
    return best_n, best_c, best_in


he = h_est["smooth belt, active (speckle)"]
yy_, xx_ = np.nonzero(np.isfinite(he))
cloud = np.c_[xx_ * 0.5, yy_ * 0.5, he[yy_, xx_]]                  # x, y in mm (0.5 mm/px on the belt), z = height
sub = cloud[np.random.default_rng(0).choice(len(cloud), min(20000, len(cloud)), replace=False)]
n_, c_, inl = fit_plane_ransac_ref(sub)
n_ = n_ if n_[2] > 0 else -n_; c_ = c_ if n_[2] > 0 else -c_
above = (cloud @ n_ + c_)                                             # height above the fitted belt plane
he_plane = np.full(he.shape, np.nan); he_plane[yy_, xx_] = above
rows = []
for m, c in zip(smasks, sl):
    core = cv2.erode(m.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
    vals = he_plane[core]
    rows.append((partsim.SCENE_CLASSES[c], HEIGHT[partsim.SCENE_CLASSES[c]], np.nanmedian(vals) if np.isfinite(vals).any() else np.nan))
print(f"belt plane normal {np.round(n_, 3)}, {inl.mean():.0%} of sampled points are belt")
print(f"{'part':<9}{'true (mm)':>10}{'measured (mm)':>15}")
for name, t, mval in rows:
    print(f"{name:<9}{t:>10.1f}{mval:>15.2f}" if np.isfinite(mval) else f"{name:<9}{t:>10.1f}{'outside range':>15}")

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Disparity to depth
Return $Z = fB/d$ as a float array, with `np.nan` wherever `d <= min_disp`.

In [ ]:
def disparity_to_depth(d, f, B, min_disp=0.0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    z = np.asarray(disparity_to_depth(np.array([120.0, 60.0, 0.0, -1.0]), 1000.0, 60.0), float)
    assert np.allclose(z[:2], [500, 1000]) and np.isnan(z[2:]).all(), f"got {z}"
    z2 = np.asarray(disparity_to_depth(np.array([10.0, 50.0]), 100.0, 1.0, min_disp=20), float)
    assert np.isnan(z2[0]) and abs(z2[1] - 2.0) < 1e-9, "disparities <= min_disp must be invalid"


check("E1 disparity_to_depth", _test_e1)

### E2. Back-projection
Given a depth map `Z` `[H, W]` (NaN = invalid) and intrinsics `f, cx, cy`, return the `[N, 3]` array of 3D points $(X, Y, Z)$ of the **valid** pixels, in row-major pixel order.

In [ ]:
def backproject(Z, f, cx, cy):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    Zt = np.array([[2.0, np.nan], [4.0, 2.0]])
    P = np.asarray(backproject(Zt, 2.0, 0.5, 0.5), float)
    assert P.shape == (3, 3), f"expected 3 valid points, got {P.shape}"
    assert np.allclose(P[0], [-0.5, -0.5, 2.0]) and np.allclose(P[1], [-1.0, 1.0, 4.0]) and np.allclose(P[2], [0.5, 0.5, 2.0]), f"got {P}"


check("E2 backproject", _test_e2)

### E3. RANSAC plane
Implement `fit_plane_ransac(points, thr=0.5, iters=200, seed=0)` returning `(n, c, inliers)` for the plane $\mathbf{n}\cdot\mathbf{p} + c = 0$ with unit normal, fitted by RANSAC with 3-point samples (use `np.random.default_rng(seed)`). Then **refine** the plane by least squares (SVD) on its inliers.

In [ ]:
def fit_plane_ransac(points, thr=0.5, iters=200, seed=0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    r = np.random.default_rng(4)
    xy = r.uniform(0, 100, (400, 2))
    z = 0.02 * xy[:, 0] - 0.01 * xy[:, 1] + 3 + r.normal(0, 0.05, 400)
    pts_ = np.c_[xy, z]
    pts_[:120, 2] += r.uniform(3, 10, 120)                      # 30% outliers above the plane
    n, c, inl = fit_plane_ransac(pts_)
    n = np.asarray(n, float); inl = np.asarray(inl, bool)
    assert abs(np.linalg.norm(n) - 1) < 1e-6, "the normal must have unit length"
    true_n = np.array([-0.02, 0.01, 1.0]); true_n /= np.linalg.norm(true_n)
    assert abs(abs(n @ true_n) - 1) < 1e-3, f"normal {n} is not parallel to the true normal {true_n}"
    assert inl[120:].mean() > 0.95 and inl[:120].mean() < 0.05, "inliers should be the plane points only"


check("E3 fit_plane_ransac", _test_e3)

### E4. Specify a 3D camera (no self-check)
A bin-picking cell must measure parts 800–1200 mm away with a depth error below 1 mm, and the matcher is good to 0.2 px. Using Section 4, find
combinations of focal length (px) and baseline (mm) that meet the requirement, and discuss what you lose with a very long baseline. Check your answer
by re-running Section 6 with `Z0 = 1000` and your parameters.

In [ ]:
# E4: your code here

## 8. Challenge: volume and flatness
Using the height map of Section 6: (1) compute each part's **volume** in mm³ (sum of heights × pixel area) and compare nuts vs washers;
(2) tilt the belt by 2° (add a linear ramp to `height`) and show that your RANSAC plane still gives correct part heights, while a fixed
"belt at 500 mm" assumption does not; (3) export the point cloud to a `.ply` file and open it in Open3D or CloudCompare.

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'After rectification, where must the match of a left pixel (x, y) lie?',
        "options": [
            'Anywhere in the right image',
            'On the same row y of the right image',
            'On the same column',
            'At (x, y)',
        ],
        "answer": '497836faa4',
        "explain": 'Rectification makes epipolar lines horizontal and aligned.',
    },
    {
        "q": 'f = 1000 px, B = 60 mm, disparity 120 px. Depth?',
        "options": [
            '50 mm',
            '500 mm',
            '7200 mm',
            '2000 mm',
        ],
        "answer": '575c6a09ce',
        "explain": 'Z = fB/d = 60000/120.',
    },
    {
        "q": 'If the distance doubles, the depth error of a stereo camera...',
        "options": [
            'halves',
            'stays the same',
            'doubles',
            'quadruples',
        ],
        "answer": '4fc7c32b39',
        "explain": 'Delta Z = Z^2 Delta d / (f B).',
    },
    {
        "q": 'Why does SGBM beat simple block matching?',
        "options": [
            'It uses colour only',
            'It adds smoothness penalties along several directions',
            'It is always faster',
            'It needs no rectification',
        ],
        "answer": 'e7e1654a35',
        "explain": 'P1 and P2 penalise disparity changes between neighbours.',
    },
    {
        "q": 'Why do active stereo cameras project a random speckle pattern?',
        "options": [
            'To measure colour',
            'To add texture to textureless surfaces so they can be matched',
            'To heat the object',
            'To calibrate the camera',
        ],
        "answer": '4981a7c928',
        "explain": 'Matching needs texture.',
    },
    {
        "q": 'What does the fundamental matrix F relate?',
        "options": [
            'Pixels to 3D points',
            'A point in one image to its epipolar line in the other',
            'Colours between cameras',
            'Depth to disparity',
        ],
        "answer": '802fd57862',
        "explain": "x'^T F x = 0.",
    },
    {
        "q": 'Which sensor gives micrometre-level height profiles of parts moving under it?',
        "options": [
            'LiDAR',
            'Laser triangulation',
            'Passive stereo',
            'ToF',
        ],
        "answer": '1f8a6b70de',
        "explain": 'A laser line + camera, one profile per frame, inline.',
    },
], title='Week 12A quiz')

## 10. Exit ticket (reflection)
1. Which 3D sensor would you choose for your project, and why (range, accuracy, surface, lighting)?
2. Why could a shiny, uniform metal part defeat passive stereo? Give two fixes.
3. What went wrong with the 2 mm washer in Section 6, and how would you measure it reliably?

## 11. What's new (2025–26)

> - **Learned stereo** (RAFT-Stereo, CREStereo, IGEV, and foundation stereo models such as FoundationStereo, 2025) handle textureless and reflective
>   surfaces far better than SGBM and generalise zero-shot; several run in real time on embedded GPUs.
> - **Industrial 3D cameras** combine active stereo or structured light with on-board neural matching and output calibrated point clouds directly.
> - **3D foundation models** (DUSt3R, MASt3R, VGGT) regress depth, camera poses and point clouds from uncalibrated image pairs or collections —
>   Session B.
> - Point-cloud deep learning (PointNet++, Point Transformer v3) and **Gaussian splatting** reconstructions are entering robotics and inspection.

## 12. References and further exploration
- Hartley, R. & Zisserman, A. (2004). *Multiple View Geometry in Computer Vision*, 2nd ed., Ch. 9–12 (epipolar geometry, triangulation).
- Zhang, Z. (2000). *A flexible new technique for camera calibration.* IEEE TPAMI 22(11). [doi:10.1109/34.888718](https://doi.org/10.1109/34.888718)
- Scharstein, D. & Szeliski, R. (2002). *A taxonomy and evaluation of dense two-frame stereo correspondence algorithms.* IJCV 47. [doi:10.1023/A:1014573219977](https://doi.org/10.1023/A:1014573219977)
- Hirschmüller, H. (2008). *Stereo processing by semiglobal matching and mutual information.* IEEE TPAMI 30(2). [doi:10.1109/TPAMI.2007.1166](https://doi.org/10.1109/TPAMI.2007.1166)
- Scharstein, D. & Pal, C. (2007). *Learning conditional random fields for stereo* (Middlebury 2006 data, incl. aloe). CVPR. [doi:10.1109/CVPR.2007.383191](https://doi.org/10.1109/CVPR.2007.383191)
- Keselman, L. et al. (2017). *Intel RealSense stereoscopic depth cameras.* CVPR Workshops. [arXiv:1705.05548](https://arxiv.org/abs/1705.05548)
- Lipson, L., Teed, Z. & Deng, J. (2021). *RAFT-Stereo.* 3DV. [arXiv:2109.07547](https://arxiv.org/abs/2109.07547)
- Wen, B. et al. (2025). *FoundationStereo: Zero-shot stereo matching.* CVPR. [arXiv:2501.09898](https://arxiv.org/abs/2501.09898)
- Szeliski (2022), Ch. 12 (depth estimation), Ch. 13 (3D reconstruction). [szeliski.org/Book](https://szeliski.org/Book/) · Open3D. [open3d.org](https://www.open3d.org/)
- OpenCV tutorials: *Epipolar geometry*, *Depth map from stereo images*. [docs.opencv.org](https://docs.opencv.org/4.x/dd/d53/tutorial_py_depthmap.html)

**Data:** `aloeL.jpg`, `aloeR.jpg`, `aloeGT.png` (Middlebury 2006) and `left01-14.jpg`, `right01-14.jpg` from the OpenCV sample data; conveyor
scene from `partsim.py`.

## 13. Key takeaways
- Two calibrated cameras + **epipolar geometry** turn matching into a 1D search after **rectification**.
- **SGBM** is the practical classical matcher; it fails on textureless, repetitive and occluded regions — **active** patterns help.
- $Z = fB/d$ and $\Delta Z \approx Z^2 \Delta d / (fB)$: specify $f$ and $B$ from the required accuracy.
- Depth maps become **point clouds**; **RANSAC planes** give reference surfaces for height and flatness measurements.

**Next session (12B):** depth from a **single** image with deep networks, and 3D foundation models.